In [ ]:
# !pip install accelerate

In [ ]:
# !pip uninstall -y bitsandbytes -q
# !pip install -U bitsandbytes

In [ ]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) y
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_http.py", line 409, in hf_raise_for_status
    response.raise_for_status()
  File "/usr/local/lib/python3.11/dist-packages/requests

In [ ]:
!git lfs install

Git LFS initialized.


In [ ]:
!git clone https://huggingface.co/Icebear-AI/Llama-2-13b-chat-arabic-lora

fatal: destination path 'Llama-2-13b-chat-arabic-lora' already exists and is not an empty directory.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/FINAL_GRAD_PROJ/TOKENIZER_FINETUNED_FINAL")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Quantization config for 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# Load quantized base model
base_model = AutoModelForCausalLM.from_pretrained(
    "/content/drive/MyDrive/FINAL_GRAD_PROJ/adaptive_base_4bit",
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16
)

# Load LoRA adapter
lora_adapter_path = "/content/drive/MyDrive/FINAL_GRAD_PROJ/adaptive_lora_adapter"  # or your path
model = PeftModel.from_pretrained(base_model, lora_adapter_path)
model.eval()


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 4096, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj)

In [ ]:
# Set system prompt (optional)
system_prompt = """أنت مساعد ذكي، مهذب، ومتعاون. يجب أن تُجيب على الأسئلة بنفس اللهجة العربية التي يستخدمها المستخدم، سواء كانت فصحى أو عامية (مثل المصرية أو الشامية أو الخليجية)."""

# Build the full prompt
user_input =  "### السؤال:\nاحسب ٣+٣ ؟\n\n### الجواب:\n"
prompt = f"[INST] <<SYS>>\n{system_prompt}\n<</SYS>>\n\n{user_input} [/INST]"

# Tokenize
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id
    )

# Decode
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n🧠 Model Response:\n", response)


🧠 Model Response:
 [INST] <<SYS>>
أنت مساعد ذكي، مهذب، ومتعاون. يجب أن تُجيب على الأسئلة بنفس اللهجة العربية التي يستخدمها المستخدم، سواء كانت فصحى أو عامية (مثل المصرية أو الشامية أو الخليجية).
<</SYS>>

### السؤال:
احسب ٣+٣ ؟

### الجواب:
 [/INST] الجواب هو ٦


##########################################################################################################################################################

In [ ]:
# !pip install -U datasets transformers peft evaluate

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, TrainingArguments,
    Trainer, BitsAndBytesConfig, DataCollatorForLanguageModeling
)
from peft import PeftModel
import torch
import evaluate
from tqdm import tqdm

In [ ]:
# === Load dataset
df = pd.read_csv("/content/drive/MyDrive/FINAL_GRAD_PROJ/formatted_new_dataset.csv")
df = df.dropna().drop_duplicates(subset=["السؤال", "الجواب"])

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

In [ ]:
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df),
})

In [ ]:
# ✅ Save test split to CSV
test_df.to_csv("/content/drive/MyDrive/FINAL_GRAD_PROJ/test_split.csv", index=False, encoding='utf-8')

print(f"✅ Test split saved to CSV!")
print(f"Test set size: {len(test_df)} examples")
print(f"Columns: {list(test_df.columns)}")

# ✅ Optional: Save all splits
train_df.to_csv("/content/drive/MyDrive/FINAL_GRAD_PROJ/train_split.csv", index=False, encoding='utf-8')
val_df.to_csv("/content/drive/MyDrive/FINAL_GRAD_PROJ/val_split.csv", index=False, encoding='utf-8')

print(f"Train set size: {len(train_df)} examples")
print(f"Validation set size: {len(val_df)} examples")

✅ Test split saved to CSV!
Test set size: 7051 examples
Columns: ['السؤال', 'الجواب']
Train set size: 56404 examples
Validation set size: 7050 examples


In [ ]:
# === Load model + adapter
model_id = "/content/drive/MyDrive/FINAL_GRAD_PROJ/adaptive_base_4bit"
adapter_id = "/content/drive/MyDrive/FINAL_GRAD_PROJ/adaptive_lora_adapter"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
model = PeftModel.from_pretrained(base_model, adapter_id)

In [ ]:
# ✅ Verify trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"✅ Trainable parameters: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")

if trainable_params == 0:
    raise ValueError("❌ No trainable parameters! Check your model setup.")

✅ Trainable parameters: 16,777,216 / 3,517,190,144 (0.48%)


In [ ]:
# === ✅ CORRECTED: Format and tokenize
def format_chat_prompt(example):
    """Format for Llama-2-chat model"""
    return f"<s>[INST] {example['السؤال'].strip()} [/INST] {example['الجواب'].strip()} </s>"

In [ ]:
def tokenize(example):
    question = example['السؤال'].strip()
    answer = example['الجواب'].strip()

    # Keep Arabic labels in the prompt
    question_tokens = tokenizer(f"###السؤال: {question}\n###الجواب:", add_special_tokens=False)["input_ids"]
    answer_tokens = tokenizer(answer, add_special_tokens=False)["input_ids"]

    total_length = len(question_tokens) + len(answer_tokens)
    max_length = 512

    if total_length > max_length:
        if len(question_tokens) > max_length - 50:  # Leave space for answer
            question_tokens = question_tokens[:max_length - 50]
        remaining_space = max_length - len(question_tokens)
        if remaining_space > 0:
            answer_tokens = answer_tokens[:remaining_space]
        else:
            answer_tokens = []

    input_ids = question_tokens + answer_tokens
    labels = [-100] * len(question_tokens) + answer_tokens  # Only learn answer part

    current_length = len(input_ids)
    if current_length < max_length:
        padding_length = max_length - current_length
        input_ids = input_ids + [tokenizer.pad_token_id] * padding_length
        labels = labels + [-100] * padding_length
        attention_mask = [1] * current_length + [0] * padding_length
    elif current_length > max_length:
        input_ids = input_ids[:max_length]
        labels = labels[:max_length]
        attention_mask = [1] * max_length
    else:
        attention_mask = [1] * max_length

    assert len(input_ids) == max_length, f"Length mismatch: {len(input_ids)} != {max_length}"
    assert len(labels) == max_length, f"Labels length mismatch: {len(labels)} != {max_length}"
    assert len(attention_mask) == max_length, f"Attention mask length mismatch: {len(attention_mask)} != {max_length}"

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [ ]:
# ✅ Apply tokenization with error handling
def safe_tokenize(example):
    try:
        return tokenize(example)
    except Exception as e:
        print(f"Error tokenizing example: {e}")
        # Return a safe default
        return {
            "input_ids": [tokenizer.pad_token_id] * 512,
            "attention_mask": [0] * 512,
            "labels": [-100] * 512
        }

tokenized_dataset = dataset.map(safe_tokenize, remove_columns=dataset["train"].column_names)

Map:   0%|          | 0/56404 [00:00<?, ? examples/s]

Map:   0%|          | 0/7050 [00:00<?, ? examples/s]

Map:   0%|          | 0/7051 [00:00<?, ? examples/s]

In [ ]:
# ✅ Verify all sequences are the same length
print("Verifying sequence lengths...")
for split_name, split_data in tokenized_dataset.items():
    lengths = [len(ex['input_ids']) for ex in split_data]
    print(f"{split_name}: min={min(lengths)}, max={max(lengths)}, all_same={len(set(lengths)) == 1}")

    if len(set(lengths)) != 1:
        print(f"⚠️  WARNING: {split_name} has variable lengths!")
        # Find problematic examples
        for i, ex in enumerate(split_data):
            if len(ex['input_ids']) != 512:
                print(f"  Example {i}: length={len(ex['input_ids'])}")
                break

Verifying sequence lengths...
train: min=512, max=512, all_same=True
validation: min=512, max=512, all_same=True
test: min=512, max=512, all_same=True


In [ ]:
# # ✅ Test the tokenization mapping before training

# def test_tokenization_mapping():
#     """Test the tokenization function with sample data"""

#     # Sample data for testing
#     test_examples = [
#         {
#             "السؤال": "ما هو الذكاء الاصطناعي؟",
#             "الجواب": "الذكاء الاصطناعي هو مجال في علوم الحاسوب يهدف إلى إنشاء أنظمة ذكية."
#         },
#         {
#             "السؤال": "اشرح مفهوم التعلم الآلي",
#             "الجواب": "التعلم الآلي هو فرع من الذكاء الاصطناعي يتعلم من البيانات."
#         },
#         {
#             "السؤال": "كيف يمكن تطبيق الذكاء الاصطناعي في الطب؟",
#             "الجواب": "يمكن تطبيق الذكاء الاصطناعي في الطب لتشخيص الأمراض وتحليل الصور الطبية."
#         }
#     ]

#     print("🧪 Testing tokenization mapping...")
#     print("=" * 60)

#     for i, example in enumerate(test_examples, 1):
#         print(f"\n📝 Example {i}:")
#         print(f"Question: {example['السؤال']}")
#         print(f"Answer: {example['الجواب']}")

#         # Test the tokenization function
#         tokenized = tokenize(example)

#         print(f"\n🔍 Tokenization Results:")
#         print(f"Input IDs length: {len(tokenized['input_ids'])}")
#         print(f"Labels length: {len(tokenized['labels'])}")
#         print(f"Attention mask length: {len(tokenized['attention_mask'])}")

#         # Decode to see the result
#         decoded_full = tokenizer.decode(tokenized['input_ids'], skip_special_tokens=True)
#         print(f"\n📄 Decoded full sequence:")
#         print(f"'{decoded_full}'")

#         # Check labels (where -100 is not present)
#         answer_labels = [label for label in tokenized['labels'] if label != -100]
#         question_labels = [label for label in tokenized['labels'] if label == -100]

#         print(f"\n🎯 Labels Analysis:")
#         print(f"Question tokens (labels = -100): {len(question_labels)}")
#         print(f"Answer tokens (labels != -100): {len(answer_labels)}")

#         # Decode only the answer part
#         if answer_labels:
#             decoded_answer = tokenizer.decode(answer_labels, skip_special_tokens=True)
#             print(f"Decoded answer part: '{decoded_answer}'")

#         print("-" * 60)

#     # Test with a few examples from your actual dataset
#     print("\n🔬 Testing with actual dataset samples...")
#     print("=" * 60)

#     for i, example in enumerate(dataset["train"].select(range(3)), 1):
#         print(f"\n�� Dataset Example {i}:")
#         print(f"Question: {example['السؤال']}")
#         print(f"Answer: {example['الجواب']}")

#         tokenized = tokenize(example)

#         # Check if lengths are reasonable
#         if len(tokenized['input_ids']) > 512:
#             print(f"⚠️  WARNING: Sequence too long ({len(tokenized['input_ids'])} tokens)")
#         else:
#             print(f"✅ Sequence length OK: {len(tokenized['input_ids'])} tokens")

#         # Check label distribution
#         answer_tokens = sum(1 for label in tokenized['labels'] if label != -100)
#         question_tokens = sum(1 for label in tokenized['labels'] if label == -100)

#         print(f"Question tokens: {question_tokens}, Answer tokens: {answer_tokens}")

#         print("-" * 60)

# # ✅ Run the test
# test_tokenization_mapping()

# # ✅ Test the complete dataset mapping
# print("\n🚀 Testing complete dataset mapping...")
# print("=" * 60)

# # Test with a small subset first
# test_subset = dataset["train"].select(range(10))
# tokenized_test = test_subset.map(tokenize, remove_columns=test_subset.column_names)

# print(f"Original dataset size: {len(test_subset)}")
# print(f"Tokenized dataset size: {len(tokenized_test)}")
# print(f"Tokenized columns: {tokenized_test.column_names}")

# # Check a few examples
# for i in range(3):
#     example = tokenized_test[i]
#     print(f"\n📋 Tokenized Example {i+1}:")
#     print(f"Input IDs: {example['input_ids'][:20]}...")  # First 20 tokens
#     print(f"Labels: {example['labels'][:20]}...")        # First 20 labels
#     print(f"Attention mask: {example['attention_mask'][:20]}...")

# print("\n✅ Tokenization mapping test completed!")

In [ ]:
# ✅ CORRECTED: Use proper data collator
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False, pad_to_multiple_of=8)

In [ ]:
# === ✅ CORRECTED: Fine-tuning
training_args = TrainingArguments(
    output_dir="./llama-finetuned",
    num_train_epochs=1,
    remove_unused_columns=False,  # ✅ Add this
    dataloader_pin_memory=False,  # ✅ Add this
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    fp16=False,  # ✅ Disable fp16
    bf16=True,
    logging_steps=10,
    save_total_limit=1,
    load_best_model_at_end=True
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: robazenhom (robazenhom-egypt-university-of-informatics) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,0.758700,0.724465


TrainOutput(global_step=7051, training_loss=0.7970110915887612, metrics={'train_runtime': 13888.1912, 'train_samples_per_second': 4.061, 'train_steps_per_second': 0.508, 'total_flos': 1.1477818718457692e+18, 'train_loss': 0.7970110915887612, 'epoch': 1.0})

In [ ]:
# ✅ Save the LoRA adapter (small file, efficient)
adapter_path = "/content/drive/MyDrive/FINAL_GRAD_PROJ/FINALE-llama-finetuned-adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f"✅ LoRA adapter saved to: {adapter_path}")
print(f"Adapter size: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} parameters")

✅ LoRA adapter saved to: /content/drive/MyDrive/FINAL_GRAD_PROJ/FINALE-llama-finetuned-adapter
Adapter size: 16,777,216 parameters


In [ ]:
# ✅ Save the complete model (larger file, includes base model)
complete_model_path = "/content/drive/MyDrive/FINAL_GRAD_PROJ/FINALE-llama-finetuned-complete"
trainer.save_model(complete_model_path)
tokenizer.save_pretrained(complete_model_path)

print(f"✅ Complete model saved to: {complete_model_path}")

✅ Complete model saved to: /content/drive/MyDrive/FINAL_GRAD_PROJ/FINALE-llama-finetuned-complete


In [ ]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=655675db5fb7a2057a776e60eeeca9500b5e8bc719175428c6aa78b2cfcf9a66
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score


In [ ]:
# ✅ Check what columns are available in each dataset
print("Original test dataset columns:", dataset["test"].column_names)
print("Tokenized test dataset columns:", tokenized_dataset["test"].column_names)

# ✅ Use the correct dataset based on available columns
if "السؤال" in dataset["test"].column_names:
    test_data = dataset["test"]
    print("Using original dataset for evaluation")
else:
    test_data = tokenized_dataset["test"]
    print("Using tokenized dataset for evaluation")
    # You'd need to decode the tokenized data back to text

Original test dataset columns: ['السؤال', 'الجواب', '__index_level_0__']
Tokenized test dataset columns: ['input_ids', 'attention_mask', 'labels']
Using original dataset for evaluation


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import pandas as pd
from datasets import Dataset

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/FINAL_GRAD_PROJ/FINALE-llama-finetuned-adapter")
base_model = AutoModelForCausalLM.from_pretrained("/content/drive/MyDrive/FINAL_GRAD_PROJ/adaptive_base_4bit", device_map="auto")
model = PeftModel.from_pretrained(base_model, "/content/drive/MyDrive/FINAL_GRAD_PROJ/FINALE-llama-finetuned-adapter")
model.eval()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['qalora_group_size', 'use_qalora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 4096, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj)

In [ ]:
import json, os
from tqdm import tqdm
import evaluate

def evaluate_and_save_progress(model, tokenizer, test_dataset, start_index=0, save_every=100, save_path="/content/drive/MyDrive/FINAL_GRAD_PROJ/eval_output.json"):
    metric = evaluate.load("rouge")
    preds, refs = [], []

    if os.path.exists(save_path):
        print("🔁 Resuming from previously saved file...")
        with open(save_path, "r", encoding="utf-8") as f:
            saved = json.load(f)
            preds = saved["predictions"]
            refs = saved["references"]
            start_index = len(preds)
            print(f"✅ Loaded {start_index} saved examples.")
    else:
        print(f"🆕 Starting fresh from index {start_index}...")

    for i in tqdm(range(start_index, len(test_dataset))):
        ex = test_dataset[i]
        question = ex["السؤال"].strip().replace("###السؤال:", "").strip()
        prompt = f"###السؤال: {question}\n###الجواب:"

        input_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).input_ids.to(model.device)

        with torch.no_grad():
            output = model.generate(
                input_ids,
                max_new_tokens=100,
                do_sample=True,
                temperature=0.7,
                pad_token_id=tokenizer.eos_token_id
            )

        decoded = tokenizer.decode(output[0], skip_special_tokens=True)
        # Extract the answer after the last '###الجواب:'
        if "###الجواب:" in decoded:
            response = decoded.split("###الجواب:")[-1].strip()
        else:
            response = decoded.strip()
        ref_answer = ex["الجواب"].strip().replace("###الجواب:", "").strip()

        preds.append(response)
        refs.append(ref_answer)

        # Save every N examples
        if (i + 1) % save_every == 0 or (i + 1) == len(test_dataset):
            with open(save_path, "w", encoding="utf-8") as f:
                json.dump({"predictions": preds, "references": refs}, f, ensure_ascii=False, indent=2)
            print(f"💾 Saved {i+1} examples to disk")

    # Final evaluation
    results = metric.compute(predictions=preds, references=refs)
    print("\n✅ Evaluation complete!")
    for k, v in results.items():
        print(f"{k}: {v:.4f}")
    return results

In [ ]:
results = evaluate_and_save_progress(model, tokenizer, dataset["test"])

🆕 Starting fresh from index 0...


  1%|▏         | 100/7051 [08:21<9:54:56,  5.14s/it]

💾 Saved 100 examples to disk


  3%|▎         | 200/7051 [16:43<9:39:51,  5.08s/it]

💾 Saved 200 examples to disk


...

 99%|█████████▉| 7000/7051 [9:48:28<04:24,  5.19s/it]

💾 Saved 7000 examples to disk


100%|██████████| 7051/7051 [9:52:48<00:00,  5.04s/it]

💾 Saved 7051 examples to disk



✅ Evaluation complete!
rouge1: 0.1046
rouge2: 0.0398
rougeL: 0.1004
rougeLsum: 0.1032
